# Descarga de imágenes LASCO C2 y C3, Evento 1 (CME halo del 17/12/2024)

**Tesis:** Cálculo cinemático y estimación volumétrica de CMEs tipo halo con SunPy

Este cuaderno usa **Fido**, la herramienta de SunPy que busca y descarga datos solares desde el
*Virtual Solar Observatory* (VSO), para bajar las imágenes FITS de los coronógrafos LASCO C2 y C3
a bordo de SOHO.

**Cómo usarlo (Windows 10):** ejecuta las celdas de arriba hacia abajo con **Shift + Enter**.
Las celdas 1 y 2 solo se necesitan la primera vez.

**Qué se corrigió respecto a la versión anterior:**
1. Se eliminó `a.Sample(20 min)`: descartaba imágenes (LASCO toma una cada ~12 min) y dejaba intervalos irregulares.
2. La ventana de tiempo empieza 1 hora antes de la CME (para tener imágenes previas que restar) y termina a las 19:30 UT.
3. Rutas portables para Windows (`Path.home()`), verificación de la descarga y resumen de los encabezados FITS.

## 1. Instalación de paquetes (solo la primera vez)

Si SunPy ya está instalado, puedes saltarte esta celda. El comando `%pip` instala en el mismo Python
que usa este cuaderno. **Después de instalar, reinicia el kernel** (menú *Kernel → Restart*).

In [1]:
# Instala SunPy con los extras de red (Fido) y de mapas, más ipywidgets (barra de progreso en Jupyter)
%pip install "sunpy[net,map]" ipywidgets

Note: you may need to restart the kernel to use updated packages.


## 2. Verificar el entorno

In [2]:
import sys                                   # información del intérprete de Python
import sunpy                                 # biblioteca principal de física solar en Python
print("Python :", sys.version.split()[0])    # versión de Python (se recomienda 3.10 o superior)
print("SunPy  :", sunpy.__version__)         # versión de SunPy instalada

Python : 3.11.5
SunPy  : 6.0.4


## 3. Parámetros del evento

In [3]:
from pathlib import Path                     # rutas de archivos que funcionan igual en Windows, Linux y Mac
from sunpy.net import Fido, attrs as a       # Fido: buscador/descargador; attrs (a): criterios de búsqueda

INICIO = "2024-12-17 15:00"                  # inicio de la ventana (UT): ~1 h antes de la primera aparición (16:00:05)
FIN = "2024-12-17 19:30"                     # fin de la ventana (UT): la CME sale del campo de C3 hacia las ~19:00
CARPETA_BASE = Path.home() / "lasco_evento1" # carpeta de destino; en Windows es C:\Users\<tu usuario>\lasco_evento1

print("Los archivos se guardarán en:", CARPETA_BASE)   # muestra la ruta completa para que sepas dónde buscarlos

Los archivos se guardarán en: C:\Users\Steve\lasco_evento1


## 4. Función de descarga

In [4]:
def descargar(detector, subcarpeta):
    """Busca y descarga todas las imágenes de un detector LASCO dentro de la ventana de tiempo."""
    carpeta = CARPETA_BASE / subcarpeta                       # carpeta de destino de este detector (c2 o c3)
    carpeta.mkdir(parents=True, exist_ok=True)                # la crea si no existe (no falla si ya existe)
    print(f"Buscando imágenes LASCO {subcarpeta.upper()} en el VSO ...")
    resultado = Fido.search(a.Time(INICIO, FIN),              # criterio 1: ventana de tiempo
                            a.Instrument.lasco,               # criterio 2: instrumento LASCO (a bordo de SOHO)
                            detector)                         # criterio 3: coronógrafo C2 o C3
    print(f"  Archivos encontrados: {resultado.file_num}")    # total de archivos que cumplen la búsqueda
    if resultado.file_num == 0:                               # si no hay nada, se avisa y se termina
        print("  No se encontró nada. Revisa tu conexión a internet y las fechas; luego vuelve a ejecutar.")
        return
    archivos = Fido.fetch(resultado, path=str(carpeta / "{file}"))   # descarga; "{file}" conserva el nombre original
    if archivos.errors:                                       # si alguna descarga falló (red, servidor, etc.)...
        print(f"  {len(archivos.errors)} descargas fallaron; reintentando ...")
        archivos = Fido.fetch(archivos)                       # ...se reintentan solo las que fallaron
    print(f"  Descargados: {len(archivos)} de {resultado.file_num}  (carpeta: {carpeta})")   # comprobación final
    if len(archivos) != resultado.file_num:                   # si faltan archivos, se avisa para no procesar datos incompletos
        print("  AVISO: faltan archivos. Vuelve a ejecutar esta celda; los ya descargados no se repiten.")

## 5. Descargar LASCO C2 (campo de visión ≈ 2 a 6 R☉)

Puede tardar varios minutos según tu conexión. Los archivos ya descargados se omiten si repites la celda.

In [5]:
descargar(a.Detector.c2, "c2")               # descarga las imágenes del coronógrafo C2

Buscando imágenes LASCO C2 en el VSO ...
  Archivos encontrados: 22


Files Downloaded:   0%|          | 0/22 [00:00<?, ?file/s]

  Descargados: 22 de 22  (carpeta: C:\Users\Steve\lasco_evento1\c2)


## 6. Descargar LASCO C3 (campo de visión ≈ 4 a 30 R☉)

In [6]:
descargar(a.Detector.c3, "c3")               # descarga las imágenes del coronógrafo C3

Buscando imágenes LASCO C3 en el VSO ...
  Archivos encontrados: 22


Files Downloaded:   0%|          | 0/22 [00:00<?, ?file/s]

  Descargados: 22 de 22  (carpeta: C:\Users\Steve\lasco_evento1\c3)


## 7. Resumen de los encabezados FITS y revisión de la cadencia

Esta celda lee **solo los encabezados** (no las imágenes) y guarda `resumen_fits.csv` con fecha, exposición,
tamaño, centro del Sol (`CRPIX`), escala (`CDELT`) y giro (`CROTA`) de cada archivo. Sirve para comprobar
que no faltan imágenes y para verificar la geometría de las imágenes.

In [7]:
import csv                                  # para escribir el resumen en un archivo CSV
from astropy.io import fits                  # lectura de archivos FITS
from astropy.time import Time                # manejo de fechas y horas astronómicas

filas = []                                                    # aquí se acumula una fila por archivo
for detector in ["c2", "c3"]:                                 # recorre las dos carpetas
    rutas = sorted(list((CARPETA_BASE / detector).glob("*.fts")) + list((CARPETA_BASE / detector).glob("*.fits")))  # todos los FITS
    for ruta in rutas:                                        # recorre cada archivo
        with fits.open(ruta) as hdul:                         # abre el FITS
            h = hdul[0].header                                # encabezado principal
        fecha, hora = str(h.get("DATE-OBS", "")), str(h.get("TIME-OBS", ""))   # LASCO guarda fecha y hora por separado
        instante = fecha if "T" in fecha else f"{fecha} {hora}".strip()        # une fecha y hora en un solo texto
        filas.append(dict(detector=detector.upper(), archivo=ruta.name,
                          instante_utc=instante.replace("/", "-"),             # formato AAAA-MM-DD HH:MM:SS
                          exptime=h.get("EXPTIME"), naxis1=h.get("NAXIS1"), naxis2=h.get("NAXIS2"),
                          crpix1=h.get("CRPIX1"), crpix2=h.get("CRPIX2"),      # centro del Sol en píxeles
                          cdelt1=h.get("CDELT1"), cdelt2=h.get("CDELT2"),      # escala en arcsec/píxel
                          crota1=h.get("CROTA1"), crota2=h.get("CROTA2")))     # giro de la imagen (0 o 180 grados)

if not filas:                                                 # si no hay archivos, no hay nada que resumir
    print("No se encontraron archivos FITS. Ejecuta primero las celdas 5 y 6.")
else:
    with open(CARPETA_BASE / "resumen_fits.csv", "w", newline="", encoding="utf-8") as f:   # abre el CSV de salida
        escritor = csv.DictWriter(f, fieldnames=list(filas[0].keys()))
        escritor.writeheader()                                # nombres de columnas
        escritor.writerows(filas)                             # todas las filas
    print(f"Resumen guardado en: {CARPETA_BASE / 'resumen_fits.csv'}")
    for detector in ["C2", "C3"]:                             # revisión de cadencia para cada detector
        t = sorted(Time(f["instante_utc"], format="iso") for f in filas if f["detector"] == detector)   # horas ordenadas
        print(f"\n{detector}: {len(t)} imágenes", end="")
        if len(t) > 1:
            huecos = [(t[i], t[i + 1]) for i in range(len(t) - 1) if (t[i + 1] - t[i]).to_value("min") > 15]  # saltos > 15 min
            print(f" | de {t[0].iso[:16]} a {t[-1].iso[:16]} UT | cadencia típica ~12 min")
            print("  Huecos mayores a 15 min:" if huecos else "  Sin huecos importantes.")
            for ini, fin in huecos:                           # lista cada hueco encontrado
                print(f"    entre {ini.iso[:16]} y {fin.iso[:16]}")

Resumen guardado en: C:\Users\Steve\lasco_evento1\resumen_fits.csv

C2: 22 imágenes | de 2024-12-17 15:12 a 2024-12-17 19:24 UT | cadencia típica ~12 min
  Sin huecos importantes.

C3: 22 imágenes | de 2024-12-17 15:06 a 2024-12-17 19:18 UT | cadencia típica ~12 min
  Sin huecos importantes.


## 8. (Opcional) Empaquetar en ZIP para compartir

Crea `muestra_evento1.zip` con solo las imágenes de una ventana corta alrededor del inicio de la CME
(cada FITS pesa ≈ 2 MB), más `resumen_fits.csv`. Ajusta `HORA_MIN` y `HORA_MAX` si quieres más o menos imágenes.

In [8]:
import zipfile                              # para crear el archivo ZIP

HORA_MIN = Time("2024-12-17 15:48")          # primera hora que se incluye en la muestra (UT)
HORA_MAX = Time("2024-12-17 17:00")          # última hora que se incluye en la muestra (UT)

ruta_zip = CARPETA_BASE.parent / "muestra_evento1.zip"        # el ZIP queda junto a la carpeta lasco_evento1
n = 0                                                         # contador de archivos incluidos
with zipfile.ZipFile(ruta_zip, "w", zipfile.ZIP_DEFLATED) as z:   # crea el ZIP comprimido
    for f in filas:                                           # recorre los archivos resumidos en la celda 7
        if HORA_MIN <= Time(f["instante_utc"], format="iso") <= HORA_MAX:    # solo los de la ventana elegida
            z.write(CARPETA_BASE / f["detector"].lower() / f["archivo"],      # archivo original en disco
                    arcname=f"{f['detector'].lower()}/{f['archivo']}")        # ruta dentro del ZIP (c2/... o c3/...)
            n += 1
    z.write(CARPETA_BASE / "resumen_fits.csv", arcname="resumen_fits.csv")    # se añade también el resumen
print(f"ZIP creado: {ruta_zip}")
print(f"Archivos FITS incluidos: {n} | tamaño: {ruta_zip.stat().st_size / 1e6:.1f} MB")

ZIP creado: C:\Users\Steve\muestra_evento1.zip
Archivos FITS incluidos: 12 | tamaño: 14.8 MB
